In [1]:
import pandas as pd

# Load all your final outputs
forecast = pd.read_csv("../reports/demand_forecast.csv")
segments = pd.read_csv("../reports/customer_segments.csv")
scorecard = pd.read_csv("../reports/supplier_scorecard.csv")
alerts = pd.read_csv("../reports/inventory_alerts.csv")

test_results = []

def check(name, condition, details=""):
    status = "PASS" if condition else "FAIL"
    test_results.append({"test": name, "status": status, "details": details})
    print(f"[{status}] {name}")

# --- Forecast checks ---
check("Forecast has no negative predicted demand",
      (forecast['predicted_qty'] >= 0).all())
check("Forecast lower_bound <= upper_bound for all rows",
      (forecast['lower_bound'] <= forecast['upper_bound']).all())
check("Forecast covers exactly 30 future days",
      len(forecast) == 30, f"actual rows: {len(forecast)}")

# --- Segmentation checks ---
check("Segment counts sum to total customer count",
      segments['segment_label'].value_counts().sum() == 179)
check("No customer appears in more than one segment",
      segments['customer_id'].duplicated().sum() == 0)

# --- Supplier scorecard checks ---
check("On-time delivery percentage is between 0 and 100",
      scorecard['on_time_pct'].between(0, 100).all())
check("Composite score is between 0 and 100",
      scorecard['composite_score'].between(0, 100).all())
check("All 14 suppliers present in scorecard",
      len(scorecard) == 14, f"actual: {len(scorecard)}")

# --- Inventory alert checks ---
check("REORDER NOW rows actually have stock <= reorder level",
      (alerts[alerts['alert_status'].str.contains("REORDER NOW", na=False)]
       .apply(lambda r: r['current_stock_qty'] <= r['reorder_level'], axis=1).all()))
check("No duplicate products in alert list",
      alerts['product_id'].duplicated().sum() == 0)

results_df = pd.DataFrame(test_results)
print(f"\n{results_df['status'].value_counts()}")
results_df.to_csv("../reports/test_results.csv", index=False)

[PASS] Forecast has no negative predicted demand
[PASS] Forecast lower_bound <= upper_bound for all rows
[PASS] Forecast covers exactly 30 future days
[PASS] Segment counts sum to total customer count
[PASS] No customer appears in more than one segment
[PASS] On-time delivery percentage is between 0 and 100
[PASS] Composite score is between 0 and 100
[PASS] All 14 suppliers present in scorecard
[PASS] REORDER NOW rows actually have stock <= reorder level
[PASS] No duplicate products in alert list

status
PASS    10
Name: count, dtype: int64
